# 02 — Feature Engineering: Pre-Earnings Technical Indicators

For each earnings event we look back at the **14 trading days** immediately before the
announcement ($T_{-14}$ to $T_{-1}$) and turn the price action into model features.

**Key design rules:**
1. **$T_{-14}..T_{-1}$ are trading days, not calendar days** — we slice by row position in the sorted series, not by date math.
2. **Indicators are computed on the FULL price history first, then sliced** — so RSI(14), SMA(20), MACD etc. are fully warmed up across the whole window (no leading NaNs).
3. **$T_{-1}$ is the last trading day STRICTLY before the announcement** — many firms report after the close, so the announcement-day bar can already contain the reaction. We exclude it to avoid leakage.

Indicators are computed with **TA-Lib** (`pip install TA-Lib`). Output: one feature row per earnings event, joined back to the target labels.

In [1]:
import os
import pandas as pd
import talib

# ── Config ──────────────────────────────────────────
DATA_DIR      = "../data/raw"
BASE_CSV      = os.path.join(DATA_DIR, "earnings_base.csv")
PRICE_DIR     = os.path.join(DATA_DIR, "ticker_prices")
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

LOOKBACK = 14   # trading days in the window: T-14 .. T-1

## Step 1 — Compute indicators on the full price series (TA-Lib)

Everything rolling/smoothed (RSI, SMA, MACD) is computed over the entire per-ticker history.
This is what guarantees the 14-day window is fully warmed up. TA-Lib works on NumPy arrays,
so we pass in the close/volume columns and assign the results back as new DataFrame columns.


In [2]:
def add_indicators(px: pd.DataFrame) -> pd.DataFrame:
    """Compute technical indicators on the FULL series (sorted ascending by date) via TA-Lib."""
    px = px.sort_values("date").reset_index(drop=True)
    close  = px["close"].to_numpy(dtype="float64")

   
    px["rsi_14"] = talib.RSI(close, timeperiod=14)

    #Simple Moving averages
    px["sma_10"] = talib.SMA(close, timeperiod=10)
    px["sma_20"] = talib.SMA(close, timeperiod=20)

    # Returns + realized volatility
    px["ret_1d"] = px["close"].pct_change()
    px["vol_20"] = px["ret_1d"].rolling(20).std()

    # MACD (12 / 26 / 9) — returns macd line, signal line, and histogram
    macd, macd_signal, _ = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    px["macd"]        = macd
    px["macd_signal"] = macd_signal

    # Volume trend vs its 20-day average
    px["vol_ratio"] = px["volume"] / px["volume"].rolling(20).mean()
    return px

## Step 2 — Slice the $T_{-14}..T_{-1}$ window and aggregate into features

We find the last trading day strictly before the announcement ($T_{-1}$), take the 14 rows
ending there, and reduce them to a single feature row. We keep both a **point-in-time snapshot
at $T_{-1}$** and **trajectory features** aggregated across the window.

In [3]:
def window_features(px: pd.DataFrame, earnings_date) -> dict | None:
    """Slice T-14..T-1 by ROW POSITION and aggregate into one feature row."""
    earnings_date = pd.Timestamp(earnings_date)

    # last trading day STRICTLY before the announcement = T-1
    prior = px[px["date"] < earnings_date]
    if len(prior) < LOOKBACK:
        return None  # not enough history before this event

    win  = prior.iloc[-LOOKBACK:]   # the 14 rows: T-14 .. T-1
    last = win.iloc[-1]             # the T-1 row

    # guard against any residual NaN in the warm-up region
    if win[["rsi_14", "sma_20", "vol_20"]].isna().any().any():
        return None

    return {
        # ── point-in-time snapshot at T-1 ──
        "rsi_14_at_T1":    last["rsi_14"],
        "macd_hist_at_T1": last["macd"] - last["macd_signal"],
        "px_vs_sma20_T1":  last["close"] / last["sma_20"] - 1,
        "px_vs_sma10_T1":  last["close"] / last["sma_10"] - 1,

        # ── trajectory aggregated over the 14-day window ──
        "ret_14d":        win["close"].iloc[-1] / win["close"].iloc[0] - 1,
        "rsi_mean":       win["rsi_14"].mean(),
        "rsi_slope":      win["rsi_14"].iloc[-1] - win["rsi_14"].iloc[0],
        "vol_mean":       win["vol_20"].mean(),
        "vol_ratio_mean": win["vol_ratio"].mean(),
        "ret_std_14d":    win["ret_1d"].std(),
    }

## Step 3 — Loop over earnings events and build the feature table

In [4]:
earnings = pd.read_csv(BASE_CSV, parse_dates=["earnings_date"])
print(f"Loaded {len(earnings)} earnings events for {earnings['ticker'].nunique()} tickers.")

price_cache = {}
rows = []
skipped = 0

for _, ev in earnings.iterrows():
    t = ev["ticker"]

    # load + compute indicators once per ticker, then reuse
    if t not in price_cache:
        path = os.path.join(PRICE_DIR, f"{t}_daily_prices.csv")
        if not os.path.exists(path):
            print(f"  -> No price file for {t}, skipping its events.")
            price_cache[t] = None
        else:
            px = pd.read_csv(path, parse_dates=["date"])
            price_cache[t] = add_indicators(px)

    px = price_cache[t]
    if px is None:
        skipped += 1
        continue

    feats = window_features(px, ev["earnings_date"])
    if feats is None:
        skipped += 1
        continue

    rows.append({"ticker": t, "earnings_date": ev["earnings_date"], **feats})

features_df = pd.DataFrame(rows)
print(f"Built features for {len(features_df)} events ({skipped} skipped for insufficient history).")
features_df.head()

Loaded 1190 earnings events for 60 tickers.
Built features for 1161 events (29 skipped for insufficient history).


,ticker,earnings_date,rsi_14_at_T1,macd_hist_at_T1,px_vs_sma20_T1,px_vs_sma10_T1,ret_14d,rsi_mean,rsi_slope,vol_mean,vol_ratio_mean,ret_std_14d
0,ABT,2026-07-16,44.965094,-0.460181,-0.023139,-0.036924,-0.035750,56.676898,-16.380157,0.018311,0.856682,0.019548
1,ABT,2026-04-16,40.109712,0.139620,-0.011579,0.002034,-0.022692,33.826997,6.012164,0.012114,0.764587,0.011144
2,ABT,2026-01-22,35.544760,-0.427644,-0.027773,-0.023571,-0.031514,46.470269,-12.869877,0.009626,1.178552,0.008589
3,ABT,2025-10-15,50.560492,-0.255856,-0.004593,0.000068,-0.000300,50.702604,-0.683466,0.010143,1.027050,0.007681
4,ABT,2025-07-17,44.762070,-0.365484,-0.011579,-0.005694,-0.010015,48.666135,-4.880467,0.012538,1.001419,0.009857


## Step 4 — Join features back to the target labels and save

In [5]:
final = earnings.merge(features_df, on=["ticker", "earnings_date"], how="inner")

print(f"Final modeling table: {final.shape[0]} rows x {final.shape[1]} cols")
print("\nFeature columns:")
print([c for c in final.columns if c not in earnings.columns])

out_path = os.path.join(PROCESSED_DIR, "features_technical.csv")
final.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")
final.head()

Final modeling table: 1161 rows x 18 cols

Feature columns:
['rsi_14_at_T1', 'macd_hist_at_T1', 'px_vs_sma20_T1', 'px_vs_sma10_T1', 'ret_14d', 'rsi_mean', 'rsi_slope', 'vol_mean', 'vol_ratio_mean', 'ret_std_14d']

Saved to: ../data/processed/features_technical.csv


,ticker,earnings_date,actual_eps,consensus_eps,fiscal_date_ending,sector_group,surprise_amount,target_label,rsi_14_at_T1,macd_hist_at_T1,px_vs_sma20_T1,px_vs_sma10_T1,ret_14d,rsi_mean,rsi_slope,vol_mean,vol_ratio_mean,ret_std_14d
0,ABT,2026-07-16,1.31,1.28,2026-06-30,healthcare,0.03,1,44.965094,-0.460181,-0.023139,-0.036924,-0.035750,56.676898,-16.380157,0.018311,0.856682,0.019548
1,ABT,2026-04-16,1.15,1.15,2026-03-31,healthcare,0.00,1,40.109712,0.139620,-0.011579,0.002034,-0.022692,33.826997,6.012164,0.012114,0.764587,0.011144
2,ABT,2026-01-22,1.50,1.49,2025-12-31,healthcare,0.01,1,35.544760,-0.427644,-0.027773,-0.023571,-0.031514,46.470269,-12.869877,0.009626,1.178552,0.008589
3,ABT,2025-10-15,1.30,1.30,2025-09-30,healthcare,0.00,1,50.560492,-0.255856,-0.004593,0.000068,-0.000300,50.702604,-0.683466,0.010143,1.027050,0.007681
4,ABT,2025-07-17,1.26,1.26,2025-06-30,healthcare,0.00,1,44.762070,-0.365484,-0.011579,-0.005694,-0.010015,48.666135,-4.880467,0.012538,1.001419,0.009857
